# Modern Portfolio Theory (MPT) - 9 ETF Indian Portfolio
**Interactive Version (Inline Outputs)**

This notebook has been adapted from the PDF-generating version to execute and display all tables, charts, and analysis directly inside Jupyter Notebook cells.

## 1. Setup & Config

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from scipy.optimize import minimize
from scipy.stats import skew, kurtosis
import warnings, os, datetime

warnings.filterwarnings('ignore')
%matplotlib inline

# CONFIG
RISK_FREE_RATE  = 0.068
START_DATE      = '2019-01-01'
END_DATE        = datetime.date.today().strftime('%Y-%m-%d')
MAX_WEIGHT      = 0.45
N_MC            = 10_000
TRADING_DAYS    = 252

TICKERS  = [
    'NIFTYBEES.NS','JUNIORBEES.NS','BANKBEES.NS','MID150BEES.NS',
    'MON100.NS','GOLDBEES.NS','SILVERBEES.NS','SETF10GILT.NS','LIQUIDBEES.NS'
]
SHORT = {
    'NIFTYBEES.NS':'NiftyBees', 'JUNIORBEES.NS':'JuniorBees', 'BANKBEES.NS':'BankBees',
    'MID150BEES.NS':'Mid150Bees', 'MON100.NS':'Nasdaq100', 'GOLDBEES.NS':'GoldBees',
    'SILVERBEES.NS':'SilverBees', 'SETF10GILT.NS':'GSec10Yr', 'LIQUIDBEES.NS':'LiquidBees'
}

PALETTE = ['#1565C0','#E53935','#F9A825','#2E7D32','#6A1B9A', '#009688', '#9E9E9E', '#795548', '#607D8B']

## 2. Data Download & Cleaning

In [ ]:
print("📥  Downloading live data from Yahoo Finance …")
raw = yf.download(TICKERS, start=START_DATE, end=END_DATE, auto_adjust=True, progress=False, group_by='column')
prices = raw['Close'].copy() if isinstance(raw.columns, pd.MultiIndex) else raw.copy()
prices.dropna(how='all', inplace=True)
prices.ffill(inplace=True); prices.dropna(inplace=True)

# Clean Dec-2019 split artifacts
prices_c = prices.copy().astype(float)
for col in prices_c.columns:
    chk = prices_c[col].pct_change()
    bad = chk.index[chk.abs() > 0.15].tolist()
    if bad:
        nan_set = set()
        for d in bad:
            pos = prices_c.index.get_loc(d)
            if pos > 0: nan_set.add(prices_c.index[pos-1])
            nan_set.add(d)
        prices_c.loc[sorted(nan_set), col] = np.nan
prices_c = prices_c.interpolate(method='time', limit=10)
prices_c.ffill(inplace=True); prices_c.bfill(inplace=True)
prices = prices_c

actual_start = prices.index[0].strftime('%Y-%m-%d')
actual_end   = prices.index[-1].strftime('%Y-%m-%d')
n_days       = len(prices)
available    = list(prices.columns)
snames       = [SHORT[t] for t in available]
n            = len(available)
print(f"✅  {n_days} trading days  |  {actual_start} → {actual_end}")

## 3. Asset Statistics

In [ ]:
# Returns & Statistics
daily = prices.pct_change().dropna()
mu_s = daily.mean() * TRADING_DAYS
cov_s = daily.cov() * TRADING_DAYS
corr_s = daily.corr()
mu = mu_s.values
Sigma = cov_s.values

def max_drawdown(price_series):
    cum = (1 + price_series.pct_change().dropna()).cumprod()
    roll_max = cum.cummax()
    dd = (cum - roll_max) / roll_max
    return dd.min()

stats_list = []
for t in available:
    dr = daily[t].dropna()
    stats_list.append({
        'ETF': SHORT[t],
        'Ann. Return (%)': mu_s[t] * 100,
        'Ann. Volatility (%)': np.sqrt(cov_s.loc[t,t]) * 100,
        'Sharpe Ratio': (mu_s[t] - RISK_FREE_RATE) / np.sqrt(cov_s.loc[t,t]),
        'Total Return (%)': ((prices[t].iloc[-1]/prices[t].iloc[0]) - 1) * 100,
        'Max Drawdown (%)': max_drawdown(prices[t]) * 100,
        'Best Day (%)': dr.max() * 100,
        'Worst Day (%)': dr.min() * 100
    })

stats_df = pd.DataFrame(stats_list)
display(stats_df.round(2))

## 4. Return Distributions

In [ ]:
# Histograms
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()
for i, t in enumerate(available):
    dr = daily[t] * 100
    axes[i].hist(dr, bins=60, color=PALETTE[i], alpha=0.8, edgecolor='white')
    axes[i].axvline(0, color='black', linestyle='--')
    axes[i].axvline(dr.mean(), color='red', linestyle=':', label=f'μ={dr.mean():.2f}%')
    axes[i].set_title(snames[i])
    axes[i].legend()

# Hide empty subplot
axes[-1].axis('off')
plt.tight_layout()
plt.show()

## 5. Correlation Matrix

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(corr_s, annot=True, cmap='RdYlGn', vmin=-1, vmax=1, xticklabels=snames, yticklabels=snames)
plt.title('Correlation Matrix')
plt.show()

## 6. Portfolio Optimization & Allocation

In [ ]:
# Optimisation
def perf(w):
    r  = float(np.dot(w, mu))
    v  = float(np.sqrt(w @ Sigma @ w))
    sr = (r - RISK_FREE_RATE)/v if v>1e-10 else 0.0
    return r, v, sr

con = [{'type':'eq','fun':lambda w: np.sum(w)-1}]
bds = tuple((0, MAX_WEIGHT) for _ in range(n))
w0  = np.array([1/n]*n)

res_sh = minimize(lambda w: -(perf(w)[2]), w0, method='SLSQP', bounds=bds, constraints=con, options={'maxiter':3000})
res_mv = minimize(lambda w: perf(w)[1], w0, method='SLSQP', bounds=bds, constraints=con, options={'maxiter':3000})

def clean_w(w):
    w[w<1e-5]=0.0; w/=w.sum(); return w

w_sh = clean_w(res_sh.x if res_sh.success else w0.copy())
w_mv = clean_w(res_mv.x if res_mv.success else w0.copy())
w_eq = clean_w(w0.copy())

r_sh,v_sh,sr_sh = perf(w_sh)
r_mv,v_mv,sr_mv = perf(w_mv)
r_eq,v_eq,sr_eq = perf(w_eq)

port_stats = pd.DataFrame([
    {'Portfolio': 'Max Sharpe', 'Return (%)': r_sh*100, 'Volatility (%)': v_sh*100, 'Sharpe': sr_sh},
    {'Portfolio': 'Min Volatility', 'Return (%)': r_mv*100, 'Volatility (%)': v_mv*100, 'Sharpe': sr_mv},
    {'Portfolio': 'Equal Weight', 'Return (%)': r_eq*100, 'Volatility (%)': v_eq*100, 'Sharpe': sr_eq}
])
display(port_stats.round(2))

allocations = pd.DataFrame({'Max Sharpe (%)': w_sh*100, 'Min Volatility (%)': w_mv*100, 'Equal Weight (%)': w_eq*100}, index=snames)
display(allocations.round(2))

## 7. Efficient Frontier

In [ ]:
# Monte Carlo
print(f"🎲 Monte Carlo ({N_MC:,}) …")
np.random.seed(42)
mc_r,mc_v,mc_sr = np.zeros(N_MC),np.zeros(N_MC),np.zeros(N_MC)
mc_w = np.zeros((N_MC, n))
for i in range(N_MC):
    w = np.clip(np.random.dirichlet(np.ones(n)), 0, MAX_WEIGHT)
    w /= w.sum(); mc_w[i]=w
    mc_r[i],mc_v[i],mc_sr[i] = perf(w)

plt.figure(figsize=(10, 6))
sc = plt.scatter(mc_v*100, mc_r*100, c=mc_sr, cmap='RdYlGn', alpha=0.35, s=2)
plt.colorbar(sc, label='Sharpe Ratio')
plt.scatter(v_sh*100, r_sh*100, s=300, color='gold', marker='*', edgecolors='black', label='Max Sharpe')
plt.scatter(v_mv*100, r_mv*100, s=150, color='blue', marker='s', edgecolors='black', label='Min Volatility')
plt.scatter(v_eq*100, r_eq*100, s=150, color='tomato', marker='^', edgecolors='black', label='Equal Weight')
plt.xlabel('Annualised Volatility (%)')
plt.ylabel('Annualised Expected Return (%)')
plt.title('Efficient Frontier')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 8. Cumulative Growth

In [ ]:
# Cumulative Growth
port_sh_cum = (1+daily[available].dot(pd.Series(w_sh,index=available))).cumprod()*100
port_mv_cum = (1+daily[available].dot(pd.Series(w_mv,index=available))).cumprod()*100
port_eq_cum = (1+daily[available].dot(pd.Series(w_eq,index=available))).cumprod()*100

plt.figure(figsize=(12, 6))
plt.plot(port_sh_cum.index, port_sh_cum, color='gold', label='Max Sharpe')
plt.plot(port_mv_cum.index, port_mv_cum, color='blue', label='Min Volatility')
plt.plot(port_eq_cum.index, port_eq_cum, color='tomato', label='Equal Weight')
plt.title('Cumulative Portfolio Growth (Base 100)')
plt.ylabel('Value')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 9. Time Series Analysis (Rolling Metrics)

In [ ]:
# Time Series Analysis: Rolling Returns and Volatility
# Rolling 252-day return & vol
roll_ret = daily.rolling(252).mean() * 252
roll_vol = daily.rolling(252).std()  * np.sqrt(252)

# Plot Rolling Returns
plt.figure(figsize=(14, 6))
for i, t in enumerate(available):
    rr = roll_ret[t].dropna() * 100
    plt.plot(rr.index, rr, color=PALETTE[i], linewidth=1.4, label=snames[i], alpha=0.85)
plt.axhline(0, color='black', linewidth=0.8, linestyle='--')
plt.axhline(RISK_FREE_RATE * 100, color='grey', linewidth=1, linestyle=':', label=f'Risk-Free {RISK_FREE_RATE:.2%}')
plt.title('Rolling 12-Month Annualised Return (%)')
plt.xlabel('Date')
plt.ylabel('Return (%)')
plt.legend(ncol=3, loc='upper right')
plt.grid(True, alpha=0.3)
plt.show()

# Plot Rolling Volatility
plt.figure(figsize=(14, 6))
for i, t in enumerate(available):
    rv = roll_vol[t].dropna() * 100
    plt.plot(rv.index, rv, color=PALETTE[i], linewidth=1.4, label=snames[i], alpha=0.85)
plt.title('Rolling 12-Month Annualised Volatility (%)')
plt.xlabel('Date')
plt.ylabel('Volatility (%)')
plt.legend(ncol=3, loc='upper right')
plt.grid(True, alpha=0.3)
plt.show()
